# 14 — Multi-Layer Perceptron for Classification

In the previous notebook, we learned how to build clean training and validation loops.

Now we will use those ideas to build and train our first complete **Multi-Layer Perceptron (MLP)** for classification.

An MLP is one of the simplest neural networks with hidden layers.

Even though modern deep learning often uses CNNs, Transformers, and other specialized architectures, MLPs are extremely important because they teach the core ideas behind neural networks:

- Input features
- Hidden layers
- Nonlinear activation functions
- Output logits
- Classification losses
- Forward propagation
- Backpropagation
- Mini-batch training
- Validation
- Generalization
- Overfitting

## In this notebook, we will study:

1. What is an MLP?
2. Input, hidden, and output layers
3. Choosing hidden dimensions
4. ReLU activations
5. Logits
6. Multi-class classification
7. Synthetic classification data
8. Complete MLP with `nn.Module`
9. Training with `DataLoader`
10. Validation loop
11. Accuracy
12. Confusion-matrix intuition
13. Decision boundaries
14. Overfitting intuition
15. Improving an MLP
16. Common MLP mistakes
17. Debugging shape problems
18. Practice exercises

## Main Goal

By the end of this notebook, you should understand this complete pipeline:

$$
\boxed{
\text{Features}
\rightarrow
\text{MLP}
\rightarrow
\text{Logits}
\rightarrow
\text{Loss}
\rightarrow
\text{Backpropagation}
\rightarrow
\text{Updated Parameters}
}
$$

and you should be able to reason about the tensor shape at every layer.


In [ ]:
import copy
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from torch.utils.data import (
    TensorDataset,
    DataLoader,
    random_split
)

print("PyTorch version:", torch.__version__)


# 1. What Is an MLP?

MLP stands for:

> **Multi-Layer Perceptron**

An MLP is a feed-forward neural network made from layers of neurons.

A simple MLP might look like:

$$
\boxed{
2
\rightarrow
16
\rightarrow
16
\rightarrow
3
}
$$

This means:

- 2 input features
- First hidden layer with 16 neurons
- Second hidden layer with 16 neurons
- 3 output neurons

Data moves from left to right.

There are no loops or recurrent connections in a standard MLP.


# 2. Why Is It Called Multi-Layer?

A simple linear classifier has one transformation:

$$
X
\rightarrow
Linear
\rightarrow
Output
$$

An MLP contains one or more hidden layers:

$$
X
\rightarrow
Linear
\rightarrow
Activation
\rightarrow
Linear
\rightarrow
Activation
\rightarrow
Output
$$

The hidden layers allow the model to build increasingly useful representations of the input.


# 3. Input Layer

The input layer corresponds to the features in one sample.

Suppose each sample contains two features:

$$
x=
\begin{array}{|c|c|}
\hline
x_1 & x_2 \\
\hline
\end{array}
$$

Then:

$$
input\_features=2
$$

For a batch of 32 samples:

$$
X.shape=(32,\ 2)
$$

The first dimension is the batch dimension.

The second dimension is the feature dimension.


# 4. Hidden Layers

A hidden layer transforms the input into a new representation.

For example:

`nn.Linear(2,16)`

maps:

$$
(batch,\ 2)
$$

to:

$$
(batch,\ 16)
$$

The 16 values are called hidden features or hidden activations.


# 5. Output Layer

For multi-class classification, the output layer usually has one output unit per class.

Suppose there are:

$$
3
$$

classes.

Then the final layer should produce:

$$
3
$$

logits per sample.

For a batch of 32:

$$
output.shape=(32,\ 3)
$$


# 6. Complete Shape Flow

For an MLP:

$$
2
\rightarrow
16
\rightarrow
16
\rightarrow
3
$$

with batch size 32:

$$
\begin{array}{|c|c|}
\hline
\textbf{Stage} & \textbf{Shape} \\
\hline
Input & (32,2) \\
\hline
Linear\ 1 & (32,16) \\
\hline
ReLU & (32,16) \\
\hline
Linear\ 2 & (32,16) \\
\hline
ReLU & (32,16) \\
\hline
Output\ layer & (32,3) \\
\hline
\end{array}
$$

Notice:

> ReLU changes values but does not change tensor shape.


# 7. Why Do We Need Activation Functions?

Suppose we stack only linear layers:

$$
Linear
\rightarrow
Linear
\rightarrow
Linear
$$

The entire network can still be simplified into one linear transformation.

So multiple linear layers alone do not give us the full power of a neural network.

We need nonlinear activation functions between layers.

A common MLP pattern is:

$$
\boxed{
Linear
\rightarrow
ReLU
\rightarrow
Linear
\rightarrow
ReLU
\rightarrow
Linear
}
$$


# 8. ReLU in an MLP

ReLU is:

$$
ReLU(x)=\max(0,x)
$$

It changes negative values to zero while leaving positive values unchanged.


In [ ]:
x = torch.tensor([
    -3.0,
    -1.0,
    0.0,
    2.0,
    5.0
])

print(
    "Input:",
    x
)

print(
    "ReLU:",
    torch.relu(x)
)


# 9. What Are Logits?

The final output of a classification network is often called:

> **Logits**

Logits are raw class scores.

For a 3-class problem, one sample might produce:

$$
\begin{array}{|c|c|c|}
\hline
2.1 & -0.4 & 1.3 \\
\hline
\end{array}
$$

These values are not probabilities.

They can be:

- Positive
- Negative
- Larger than 1


# 10. From Logits to Predicted Class

For multi-class classification, the predicted class is usually the index of the largest logit.

Use:

```python
predicted_class = logits.argmax(dim=1)
```


In [ ]:
logits = torch.tensor([
    [2.1, -0.4, 1.3],
    [0.2, 3.0, 1.1]
])

predictions = logits.argmax(
    dim=1
)

print(
    "Predicted classes:",
    predictions
)


# 11. Softmax for Probabilities

If probabilities are needed, apply softmax across the class dimension.


In [ ]:
probabilities = torch.softmax(
    logits,
    dim=1
)

print(probabilities)

print(
    "Row sums:",
    probabilities.sum(dim=1)
)


For training with:

`nn.CrossEntropyLoss()`

do **not** apply softmax before the loss.

Use raw logits:

```python
loss = criterion(
    logits,
    targets
)
```


# 12. Multi-Class Classification Setup

For:

$$
C=3
$$

classes:

Model output:

$$
(batch,\ 3)
$$

Targets:

$$
(batch)
$$

Targets contain integer class indices:

$$
0,\ 1,\ 2
$$

Target dtype:

`torch.long`

Loss:

`nn.CrossEntropyLoss()`


# 13. Creating Synthetic Classification Data

We will create a 2D, 3-class classification dataset.

Because the input has only two features, we can visualize:

- The data
- The model's decision boundaries

We will generate three clusters.


In [ ]:
torch.manual_seed(42)

samples_per_class = 250

center_0 = torch.tensor([
    -2.0,
    -1.5
])

center_1 = torch.tensor([
    2.0,
    -1.0
])

center_2 = torch.tensor([
    0.0,
    2.5
])

class_0 = (
    torch.randn(
        samples_per_class,
        2
    )
    * 0.9
    + center_0
)

class_1 = (
    torch.randn(
        samples_per_class,
        2
    )
    * 0.9
    + center_1
)

class_2 = (
    torch.randn(
        samples_per_class,
        2
    )
    * 0.9
    + center_2
)

features = torch.cat(
    [
        class_0,
        class_1,
        class_2
    ],
    dim=0
)

targets = torch.cat(
    [
        torch.zeros(
            samples_per_class,
            dtype=torch.long
        ),
        torch.ones(
            samples_per_class,
            dtype=torch.long
        ),
        torch.full(
            (samples_per_class,),
            2,
            dtype=torch.long
        )
    ]
)

print(
    "Features:",
    features.shape
)

print(
    "Targets:",
    targets.shape
)


# 14. Visualizing the Dataset

Each point has:

- Feature 1
- Feature 2
- A class label


In [ ]:
plt.figure(figsize=(8, 6))

for class_index in range(3):
    mask = (
        targets
        == class_index
    )

    plt.scatter(
        features[mask, 0].numpy(),
        features[mask, 1].numpy(),
        label=f"Class {class_index}",
        alpha=0.7
    )

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Synthetic 3-Class Dataset")
plt.legend()
plt.show()


# 15. Why Synthetic Data Is Useful

Synthetic data lets us understand the full training pipeline without worrying about:

- File loading
- Image decoding
- Complex preprocessing
- Large datasets

The model concepts are the same.

Later, the same pipeline will be applied to real data.


# 16. Shuffle Before Splitting

Our generated data is currently arranged class-by-class.

We should shuffle indices before creating train and validation splits.


In [ ]:
torch.manual_seed(42)

permutation = torch.randperm(
    features.size(0)
)

features = features[
    permutation
]

targets = targets[
    permutation
]

print(
    targets[:20]
)


# 17. Creating Train and Validation Sets

We will use:

- 80% training
- 20% validation


In [ ]:
dataset = TensorDataset(
    features,
    targets
)

train_size = int(
    0.8
    * len(dataset)
)

val_size = (
    len(dataset)
    - train_size
)

train_dataset, val_dataset = random_split(
    dataset,
    [
        train_size,
        val_size
    ],
    generator=(
        torch.Generator()
        .manual_seed(42)
    )
)

print(
    "Training samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)


# 18. Creating DataLoaders


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)


# 19. Inspecting One Batch


In [ ]:
batch_features, batch_targets = next(
    iter(train_loader)
)

print(
    "Feature batch:",
    batch_features.shape
)

print(
    "Target batch:",
    batch_targets.shape
)

print(
    "Target dtype:",
    batch_targets.dtype
)


The shapes should be:

$$
features=(batch,\ 2)
$$

and:

$$
targets=(batch)
$$

This matches the requirements of our MLP and `CrossEntropyLoss`.


# 20. Building the MLP With `nn.Module`

Let's build:

$$
2
\rightarrow
32
\rightarrow
16
\rightarrow
3
$$


In [ ]:
class MLPClassifier(nn.Module):
    def __init__(
        self,
        input_dim=2,
        hidden_dim_1=32,
        hidden_dim_2=16,
        num_classes=3
    ):
        super().__init__()

        self.fc1 = nn.Linear(
            input_dim,
            hidden_dim_1
        )

        self.fc2 = nn.Linear(
            hidden_dim_1,
            hidden_dim_2
        )

        self.fc3 = nn.Linear(
            hidden_dim_2,
            num_classes
        )

        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)

        x = self.fc2(x)
        x = self.relu(x)

        x = self.fc3(x)

        return x

model = MLPClassifier()

print(model)


# 21. Understanding the Model Architecture

The first layer:

`nn.Linear(2,32)`

maps:

$$
(batch,\ 2)
\rightarrow
(batch,\ 32)
$$

Second layer:

`nn.Linear(32,16)`

maps:

$$
(batch,\ 32)
\rightarrow
(batch,\ 16)
$$

Final layer:

`nn.Linear(16,3)`

maps:

$$
(batch,\ 16)
\rightarrow
(batch,\ 3)
$$


# 22. Forward-Pass Shape Inspection


In [ ]:
sample_batch = torch.randn(
    8,
    2
)

x1 = model.fc1(
    sample_batch
)

a1 = model.relu(
    x1
)

x2 = model.fc2(
    a1
)

a2 = model.relu(
    x2
)

logits = model.fc3(
    a2
)

print(
    "Input:",
    sample_batch.shape
)

print(
    "After fc1:",
    x1.shape
)

print(
    "After ReLU 1:",
    a1.shape
)

print(
    "After fc2:",
    x2.shape
)

print(
    "After ReLU 2:",
    a2.shape
)

print(
    "Logits:",
    logits.shape
)


# 23. Choosing Hidden Dimensions

There is no universal best hidden size.

A hidden dimension controls the width of a hidden representation.

Examples:

$$
2
\rightarrow
8
\rightarrow
3
$$

is a small network.

$$
2
\rightarrow
128
\rightarrow
128
\rightarrow
3
$$

is much larger.

Larger hidden layers:

- Increase model capacity
- Increase parameter count
- Increase computation
- Can increase overfitting risk

The best size depends on:

- Data complexity
- Dataset size
- Noise
- Regularization
- Validation performance


# 24. Counting Parameters

Let's count model parameters.


In [ ]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Total parameters:",
    total_parameters
)


For our model:

First layer:

$$
2\times32+32
$$

Second layer:

$$
32\times16+16
$$

Third layer:

$$
16\times3+3
$$

The total is the sum of all weights and biases.


# 25. Parameter Shapes


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        tuple(parameter.shape)
    )


# 26. Loss Function

Because this is a multi-class problem with one correct class per sample, we use:

`nn.CrossEntropyLoss()`


In [ ]:
criterion = nn.CrossEntropyLoss()

print(criterion)


# 27. Optimizer

We will begin with Adam.


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

print(optimizer)


# 28. Device Setup


In [ ]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(
    device
)

print(
    "Using device:",
    device
)


# 29. One Training Batch

Let's run one complete optimization step.


In [ ]:
model.train()

batch_features, batch_targets = next(
    iter(train_loader)
)

batch_features = batch_features.to(
    device
)

batch_targets = batch_targets.to(
    device
)

optimizer.zero_grad()

logits = model(
    batch_features
)

loss = criterion(
    logits,
    batch_targets
)

loss.backward()

optimizer.step()

print(
    "Logits shape:",
    logits.shape
)

print(
    "Loss:",
    loss.item()
)


# 30. Training Accuracy for One Batch


In [ ]:
with torch.no_grad():
    predictions = logits.argmax(
        dim=1
    )

    batch_accuracy = (
        predictions
        == batch_targets
    ).float().mean()

print(
    "Batch accuracy:",
    batch_accuracy.item()
)


# 31. Reusable Training Function


In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    device
):
    model.train()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    for inputs, targets in loader:
        inputs = inputs.to(
            device
        )

        targets = targets.to(
            device
        )

        optimizer.zero_grad()

        logits = model(
            inputs
        )

        loss = criterion(
            logits,
            targets
        )

        loss.backward()

        optimizer.step()

        batch_size = (
            targets.size(0)
        )

        total_loss += (
            loss.item()
            * batch_size
        )

        predictions = logits.argmax(
            dim=1
        )

        total_correct += (
            predictions
            == targets
        ).sum().item()

        total_samples += (
            batch_size
        )

    epoch_loss = (
        total_loss
        / total_samples
    )

    epoch_accuracy = (
        total_correct
        / total_samples
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 32. Reusable Validation Function


In [ ]:
def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            targets = targets.to(
                device
            )

            logits = model(
                inputs
            )

            loss = criterion(
                logits,
                targets
            )

            batch_size = (
                targets.size(0)
            )

            total_loss += (
                loss.item()
                * batch_size
            )

            predictions = logits.argmax(
                dim=1
            )

            total_correct += (
                predictions
                == targets
            ).sum().item()

            total_samples += (
                batch_size
            )

    epoch_loss = (
        total_loss
        / total_samples
    )

    epoch_accuracy = (
        total_correct
        / total_samples
    )

    return (
        epoch_loss,
        epoch_accuracy
    )


# 33. Training the MLP

We will train for several epochs and save the best validation model.


In [ ]:
torch.manual_seed(42)

model = MLPClassifier().to(
    device
)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

epochs = 40

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss = float("inf")

best_state = copy.deepcopy(
    model.state_dict()
)

for epoch in range(epochs):
    train_loss, train_accuracy = (
        train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )
    )

    val_loss, val_accuracy = (
        validate_one_epoch(
            model,
            val_loader,
            criterion,
            device
        )
    )

    history[
        "train_loss"
    ].append(
        train_loss
    )

    history[
        "train_accuracy"
    ].append(
        train_accuracy
    )

    history[
        "val_loss"
    ].append(
        val_loss
    )

    history[
        "val_accuracy"
    ].append(
        val_accuracy
    )

    if val_loss < best_val_loss:
        best_val_loss = (
            val_loss
        )

        best_state = copy.deepcopy(
            model.state_dict()
        )

    if (
        epoch == 0
        or
        (epoch + 1) % 5 == 0
    ):
        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.3f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.3f}"
        )

model.load_state_dict(
    best_state
)


# 34. Plotting Loss Curves


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_loss"],
    label="Training loss"
)

plt.plot(
    history["val_loss"],
    label="Validation loss"
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")
plt.title("MLP Training and Validation Loss")
plt.legend()
plt.show()


# 35. Plotting Accuracy Curves


In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    history["train_accuracy"],
    label="Training accuracy"
)

plt.plot(
    history["val_accuracy"],
    label="Validation accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("MLP Training and Validation Accuracy")
plt.legend()
plt.show()


# 36. Final Validation Performance


In [ ]:
val_loss, val_accuracy = validate_one_epoch(
    model,
    val_loader,
    criterion,
    device
)

print(
    "Validation loss:",
    val_loss
)

print(
    "Validation accuracy:",
    val_accuracy
)


# 37. Collecting Validation Predictions

To study mistakes, we need:

- True labels
- Predicted labels

for every validation sample.


In [ ]:
def collect_predictions(
    model,
    loader,
    device
):
    model.eval()

    all_targets = []
    all_predictions = []
    all_probabilities = []

    with torch.no_grad():
        for inputs, targets in loader:
            inputs = inputs.to(
                device
            )

            logits = model(
                inputs
            )

            probabilities = torch.softmax(
                logits,
                dim=1
            )

            predictions = logits.argmax(
                dim=1
            )

            all_targets.append(
                targets.cpu()
            )

            all_predictions.append(
                predictions.cpu()
            )

            all_probabilities.append(
                probabilities.cpu()
            )

    return (
        torch.cat(all_targets),
        torch.cat(all_predictions),
        torch.cat(all_probabilities)
    )

val_true, val_pred, val_prob = collect_predictions(
    model,
    val_loader,
    device
)

print(
    "Targets:",
    val_true.shape
)

print(
    "Predictions:",
    val_pred.shape
)

print(
    "Probabilities:",
    val_prob.shape
)


# 38. Confusion Matrix Intuition

Accuracy tells us:

> How many predictions were correct overall?

A confusion matrix tells us:

> Which classes are being confused with which other classes?

For 3 classes:

$$
\begin{array}{c|c|c|c}
 & Pred\ 0 & Pred\ 1 & Pred\ 2 \\
\hline
True\ 0 & ? & ? & ? \\
\hline
True\ 1 & ? & ? & ? \\
\hline
True\ 2 & ? & ? & ? \\
\end{array}
$$

Rows represent true classes.

Columns represent predicted classes.


# 39. Building a Confusion Matrix Manually


In [ ]:
def confusion_matrix_torch(
    targets,
    predictions,
    num_classes
):
    matrix = torch.zeros(
        num_classes,
        num_classes,
        dtype=torch.long
    )

    for true_label, pred_label in zip(
        targets,
        predictions
    ):
        matrix[
            true_label.long(),
            pred_label.long()
        ] += 1

    return matrix

confusion = confusion_matrix_torch(
    val_true,
    val_pred,
    num_classes=3
)

print(confusion)


# 40. Reading the Confusion Matrix

The diagonal contains correct predictions:

$$
(0,0),\ (1,1),\ (2,2)
$$

Off-diagonal values are errors.

For example:

$$
matrix[1,2]
$$

means:

> True class was 1, but the model predicted class 2.


# 41. Visualizing the Confusion Matrix


In [ ]:
plt.figure(figsize=(6, 5))

plt.imshow(
    confusion.numpy()
)

plt.title("Validation Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")

plt.xticks(
    range(3)
)

plt.yticks(
    range(3)
)

for row in range(3):
    for col in range(3):
        plt.text(
            col,
            row,
            int(
                confusion[
                    row,
                    col
                ]
            ),
            ha="center",
            va="center"
        )

plt.colorbar()
plt.show()


# 42. Per-Class Accuracy

A confusion matrix can also help us compute class-specific performance.

For class $i$:

$$
class\ accuracy_i
=
\frac{
correct\ predictions\ for\ class\ i
}{
number\ of\ true\ samples\ in\ class\ i
}
$$


In [ ]:
for class_index in range(3):
    class_total = (
        confusion[
            class_index
        ].sum().item()
    )

    class_correct = (
        confusion[
            class_index,
            class_index
        ].item()
    )

    class_accuracy = (
        class_correct
        / class_total
        if class_total > 0
        else 0.0
    )

    print(
        f"Class {class_index} accuracy: "
        f"{class_accuracy:.3f}"
    )


# 43. Decision Boundary Intuition

A classifier divides feature space into regions.

For our 2D dataset, every point:

$$
(x_1,x_2)
$$

is assigned to one of the three classes.

The borders between these regions are called:

> **Decision boundaries**

Because the input has only two features, we can visualize them.


# 44. Creating a Grid of 2D Points

We create many points covering the visible feature space.

Then the model predicts a class for every point.


In [ ]:
x_min = (
    features[:, 0].min().item()
    - 1.0
)

x_max = (
    features[:, 0].max().item()
    + 1.0
)

y_min = (
    features[:, 1].min().item()
    - 1.0
)

y_max = (
    features[:, 1].max().item()
    + 1.0
)

grid_x, grid_y = torch.meshgrid(
    torch.linspace(
        x_min,
        x_max,
        250
    ),
    torch.linspace(
        y_min,
        y_max,
        250
    ),
    indexing="xy"
)

grid_points = torch.stack(
    [
        grid_x.reshape(-1),
        grid_y.reshape(-1)
    ],
    dim=1
)

print(
    "Grid points:",
    grid_points.shape
)


# 45. Predicting the Grid


In [ ]:
model.eval()

with torch.no_grad():
    grid_logits = model(
        grid_points.to(
            device
        )
    )

    grid_predictions = (
        grid_logits.argmax(
            dim=1
        )
        .cpu()
    )

grid_predictions = (
    grid_predictions.reshape(
        grid_x.shape
    )
)

print(
    grid_predictions.shape
)


# 46. Plotting the Decision Regions


In [ ]:
plt.figure(figsize=(8, 6))

plt.contourf(
    grid_x.numpy(),
    grid_y.numpy(),
    grid_predictions.numpy(),
    alpha=0.25
)

for class_index in range(3):
    mask = (
        targets
        == class_index
    )

    plt.scatter(
        features[mask, 0].numpy(),
        features[mask, 1].numpy(),
        label=f"Class {class_index}",
        alpha=0.7
    )

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("MLP Decision Regions")
plt.legend()
plt.show()


# 47. Why an MLP Can Learn Nonlinear Boundaries

Each linear layer creates a linear transformation.

ReLU introduces nonlinearity.

By stacking:

$$
Linear
\rightarrow
ReLU
\rightarrow
Linear
\rightarrow
ReLU
\rightarrow
Linear
$$

the network can build piecewise nonlinear decision boundaries.

This is much more flexible than a single linear classifier.


# 48. A Linear Classifier for Comparison

Let's train a model with no hidden layers:

$$
2
\rightarrow
3
$$


In [ ]:
linear_model = nn.Linear(
    2,
    3
).to(
    device
)

linear_criterion = (
    nn.CrossEntropyLoss()
)

linear_optimizer = torch.optim.Adam(
    linear_model.parameters(),
    lr=0.01
)

for _ in range(40):
    train_one_epoch(
        linear_model,
        train_loader,
        linear_criterion,
        linear_optimizer,
        device
    )

linear_val_loss, linear_val_accuracy = (
    validate_one_epoch(
        linear_model,
        val_loader,
        linear_criterion,
        device
    )
)

print(
    "Linear validation accuracy:",
    linear_val_accuracy
)

print(
    "MLP validation accuracy:",
    val_accuracy
)


Our current synthetic blobs are fairly easy, so even a linear classifier may perform well.

The deeper lesson is:

> MLPs become especially valuable when class boundaries are nonlinear.


# 49. Creating a Nonlinear XOR-Style Dataset

To see why hidden layers matter, let's create a simple XOR-style pattern.

Class 1 when the two signs differ.

Class 0 when the signs match.


In [ ]:
torch.manual_seed(123)

xor_features = torch.randn(
    800,
    2
)

xor_targets = (
    (
        xor_features[:, 0] > 0
    )
    !=
    (
        xor_features[:, 1] > 0
    )
).long()

print(
    xor_features.shape,
    xor_targets.shape
)


In [ ]:
plt.figure(figsize=(7, 6))

for class_index in range(2):
    mask = (
        xor_targets
        == class_index
    )

    plt.scatter(
        xor_features[mask, 0].numpy(),
        xor_features[mask, 1].numpy(),
        label=f"Class {class_index}",
        alpha=0.6
    )

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Nonlinear XOR-Style Data")
plt.legend()
plt.show()


A single straight decision boundary cannot separate all four regions correctly.

An MLP with nonlinear hidden layers can learn this kind of pattern.


# 50. Choosing Model Capacity

Model capacity describes how flexible the model is.

Capacity increases with things such as:

- More hidden layers
- More hidden neurons
- More parameters

Too little capacity:

> **Underfitting**

Too much capacity relative to data:

> Increased risk of **overfitting**

The goal is not to create the largest model.

The goal is to create a model that generalizes well.


# 51. Underfitting Intuition

A model may underfit when it is too simple to capture the pattern.

Typical signs:

- Training loss remains high
- Training accuracy remains low
- Validation accuracy is also low

Possible improvements:

- Increase model capacity
- Train longer
- Improve optimization
- Improve features
- Use a better architecture


# 52. Overfitting Intuition

Overfitting happens when a model learns training-specific details that do not generalize well.

A common pattern is:

- Training loss continues decreasing
- Training accuracy becomes very high
- Validation loss stops improving or gets worse

The gap between training and validation performance grows.


# 53. Example Overfitting Pattern

Conceptually:

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Epoch} & \textbf{Train Loss} & \textbf{Val Loss} \\
\hline
1 & 1.0 & 1.1 \\
\hline
10 & 0.4 & 0.5 \\
\hline
20 & 0.2 & 0.45 \\
\hline
30 & 0.05 & 0.70 \\
\hline
\end{array}
$$

The best validation model may have occurred before the final epoch.

This is why checkpointing and early stopping matter.


# 54. Ways to Improve an MLP

Possible improvements include:

1. Better feature scaling
2. Better hidden dimensions
3. More or fewer layers
4. Better learning rate
5. More training data
6. Weight decay
7. Dropout
8. Early stopping
9. Better initialization
10. Better task-specific preprocessing

Do not change everything at once.

Make controlled experiments.


# 55. Input Normalization

MLPs often train better when input features are on comparable scales.

For each feature, a common standardization is:

$$
x_{norm}
=
\frac{x-\mu}{\sigma}
$$

where:

- $\mu$ = training-set mean
- $\sigma$ = training-set standard deviation

Important:

> Compute normalization statistics from the training data, not from validation or test data.


In [ ]:
train_indices = (
    train_dataset.indices
)

train_feature_tensor = (
    dataset.tensors[0][
        train_indices
    ]
)

feature_mean = (
    train_feature_tensor.mean(
        dim=0
    )
)

feature_std = (
    train_feature_tensor.std(
        dim=0
    )
)

print(
    "Training mean:",
    feature_mean
)

print(
    "Training std:",
    feature_std
)


# 56. Why Train-Only Statistics Matter

If we compute normalization statistics using validation or test data, information from those datasets influences preprocessing.

That is a subtle form of data leakage.

A good workflow is:

1. Split data
2. Compute statistics on training split
3. Apply the same training-derived transformation to validation/test


# 57. Dropout

Dropout randomly sets some hidden activations to zero during training.

Example:

```python
nn.Dropout(p=0.5)
```

It can reduce overfitting in some models.

During evaluation:

`model.eval()`

disables dropout randomness.


In [ ]:
dropout_model = nn.Sequential(
    nn.Linear(2, 32),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

print(dropout_model)


# 58. Weight Decay

Weight decay is another regularization technique.

Example:

```python
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)
```

It discourages unnecessarily large parameter values.


In [ ]:
regularized_model = MLPClassifier().to(
    device
)

regularized_optimizer = torch.optim.Adam(
    regularized_model.parameters(),
    lr=0.01,
    weight_decay=1e-4
)

print(regularized_optimizer)


# 59. Early Stopping

Early stopping monitors validation performance and stops training when improvement stops for several epochs.

This can:

- Save computation
- Reduce unnecessary overfitting
- Restore a strong validation checkpoint

We already built early-stopping intuition in the previous notebook.


# 60. More Data

One of the strongest ways to improve generalization is often:

> **More representative training data**

If the training distribution is too small or narrow, a large network cannot invent missing real-world variability.

For medical imaging, diversity can include:

- Patients
- Devices
- Sites
- Operators
- Acquisition settings
- Demographics

Data quality can matter as much as model architecture.


# 61. Deeper Is Not Automatically Better

A deeper MLP can represent more complex functions.

But deeper models can also:

- Be harder to optimize
- Overfit
- Use more computation
- Require more tuning

Always compare models using validation performance.


# 62. Common Mistake — Applying Softmax Before `CrossEntropyLoss`

Incorrect:

```python
probabilities = torch.softmax(
    logits,
    dim=1
)

loss = criterion(
    probabilities,
    targets
)
```

Correct:

```python
loss = criterion(
    logits,
    targets
)
```

`CrossEntropyLoss` expects raw logits.


# 63. Common Mistake — Wrong Output Dimension

For 3 classes:

Final layer should normally be:

`nn.Linear(hidden_dim, 3)`

not:

`nn.Linear(hidden_dim, 1)`

when using standard multi-class `CrossEntropyLoss`.


# 64. Common Mistake — Wrong Target Shape

For standard multi-class classification:

Logits:

$$
(batch,\ classes)
$$

Targets:

$$
(batch)
$$

not:

$$
(batch,\ 1)
$$

for the usual class-index form.


# 65. Common Mistake — Wrong Target Dtype

For class-index `CrossEntropyLoss` targets:

Use:

`torch.long`

not floating-point class labels.


# 66. Common Mistake — Forgetting Nonlinearity

This network:

```python
Linear
Linear
Linear
```

without activations is still equivalent to one linear transformation.

For a true MLP, hidden layers usually need nonlinear activations.


# 67. Common Mistake — Making the Model Huge Immediately

A huge model can make debugging harder.

A better process is:

1. Start simple
2. Confirm the pipeline works
3. Establish a baseline
4. Change one design choice at a time


# 68. Common Mistake — Looking Only at Training Accuracy

High training accuracy does not guarantee generalization.

Always monitor validation performance.


# 69. Common Mistake — Using the Test Set Repeatedly

The test set should not guide architecture choices.

Use:

- Training data for learning
- Validation data for model selection
- Test data for final evaluation


# 70. MLP Debugging Checklist

If the model does not train correctly, inspect:

1. Input shape
2. Target shape
3. Target dtype
4. Number of output logits
5. Loss function
6. Hidden activation placement
7. Learning rate
8. Gradient existence
9. Parameter updates
10. Training loss
11. Validation loss
12. Training accuracy
13. Validation accuracy
14. Data leakage
15. Class distribution


In [ ]:
for name, parameter in model.named_parameters():
    print(
        name,
        "| shape:",
        tuple(parameter.shape),
        "| grad exists:",
        parameter.grad is not None
    )


# 71. Checking Class Distribution

Before training a classifier, inspect the number of samples in each class.


In [ ]:
class_counts = torch.bincount(
    targets
)

print(
    "Class counts:",
    class_counts
)


Strong class imbalance may require changes such as:

- Class-weighted loss
- Sampling strategies
- Threshold tuning for binary problems
- Better metrics

Accuracy alone may be misleading on imbalanced datasets.


# 72. Shape Reasoning Example

Suppose:

$$
batch=64
$$

and:

$$
model:
10
\rightarrow
128
\rightarrow
64
\rightarrow
5
$$

Then:

$$
\begin{array}{|c|c|}
\hline
\textbf{Stage} & \textbf{Shape} \\
\hline
Input & (64,10) \\
\hline
Hidden\ 1 & (64,128) \\
\hline
ReLU & (64,128) \\
\hline
Hidden\ 2 & (64,64) \\
\hline
ReLU & (64,64) \\
\hline
Logits & (64,5) \\
\hline
\end{array}
$$

Targets for `CrossEntropyLoss`:

$$
(64)
$$


# 73. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Build an MLP:

$$
4
\rightarrow
16
\rightarrow
3
$$

for 3-class classification.

## Exercise 2

What is the output shape for:

$$
batch=32
$$

using the model from Exercise 1?

## Exercise 3

Build:

$$
10
\rightarrow
64
\rightarrow
32
\rightarrow
5
$$

with ReLU activations.

## Exercise 4

Count the total parameters in:

$$
4
\rightarrow
16
\rightarrow
3
$$

## Exercise 5

Create synthetic 2D data for 3 classes.

## Exercise 6

Train an MLP using:

- `DataLoader`
- `CrossEntropyLoss`
- Adam
- Validation loop

## Exercise 7

Compute validation accuracy.

## Exercise 8

Build a confusion matrix manually.

## Exercise 9

Plot a 2D decision boundary.

## Exercise 10

Add dropout to the hidden layers and compare validation performance.


# 74. Conceptual Challenges

Answer before running code.

## Challenge 1

Why does an MLP need nonlinear activation functions?

## Challenge 2

For 7 classes, how many output logits should the final layer produce?

## Challenge 3

Why do we use raw logits with `CrossEntropyLoss`?

## Challenge 4

Why can training accuracy increase while validation accuracy decreases?

## Challenge 5

What information does a confusion matrix provide that accuracy does not?

## Challenge 6

Why can a decision boundary from an MLP be nonlinear?

## Challenge 7

Why is a larger hidden dimension not always better?

## Challenge 8

Why should normalization statistics come only from training data?

## Challenge 9

Why can dropout help generalization?

## Challenge 10

Why should model improvements be evaluated on validation data rather than test data?


# 75. Exercise Solutions


In [ ]:
# Exercise 1
exercise_model_1 = nn.Sequential(
    nn.Linear(4, 16),
    nn.ReLU(),
    nn.Linear(16, 3)
)

print(
    "Exercise 1:"
)

print(
    exercise_model_1
)

# Exercise 2
exercise_input = torch.randn(
    32,
    4
)

exercise_output = exercise_model_1(
    exercise_input
)

print(
    "Exercise 2 output shape:",
    exercise_output.shape
)

# Exercise 3
exercise_model_3 = nn.Sequential(
    nn.Linear(10, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
    nn.ReLU(),
    nn.Linear(32, 5)
)

print(
    "Exercise 3:"
)

print(
    exercise_model_3
)

# Exercise 4
exercise_parameter_count = sum(
    p.numel()
    for p in exercise_model_1.parameters()
)

print(
    "Exercise 4 parameters:",
    exercise_parameter_count
)

# Exercise 5
torch.manual_seed(7)

ex_class_0 = (
    torch.randn(100, 2)
    + torch.tensor([-2.0, 0.0])
)

ex_class_1 = (
    torch.randn(100, 2)
    + torch.tensor([2.0, 0.0])
)

ex_class_2 = (
    torch.randn(100, 2)
    + torch.tensor([0.0, 2.5])
)

ex_features = torch.cat(
    [
        ex_class_0,
        ex_class_1,
        ex_class_2
    ]
)

ex_targets = torch.cat(
    [
        torch.zeros(
            100,
            dtype=torch.long
        ),
        torch.ones(
            100,
            dtype=torch.long
        ),
        torch.full(
            (100,),
            2,
            dtype=torch.long
        )
    ]
)

print(
    "Exercise 5:",
    ex_features.shape,
    ex_targets.shape
)


# 76. Key Takeaways

In this notebook, we learned:

- What an MLP is
- Input layers
- Hidden layers
- Output layers
- Hidden dimensions
- ReLU activations
- Logits
- Softmax for inference
- Multi-class classification
- Synthetic classification data
- `nn.Module` MLP implementation
- DataLoader-based training
- Validation loops
- Accuracy
- Confusion matrices
- Per-class accuracy
- Decision boundaries
- Linear classifier comparison
- Nonlinear XOR intuition
- Model capacity
- Underfitting
- Overfitting
- Input normalization
- Dropout
- Weight decay
- Early stopping
- Common MLP mistakes
- MLP debugging

The central MLP pattern is:

$$
\boxed{
Input
\rightarrow
Linear
\rightarrow
ReLU
\rightarrow
Linear
\rightarrow
ReLU
\rightarrow
Logits
}
$$

For multi-class classification:

$$
\boxed{
Logits
+
Class\ Indices
\rightarrow
CrossEntropyLoss
}
$$


# 77. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What does MLP stand for?
2. What is the role of the input layer?
3. What is a hidden layer?
4. What does hidden dimension mean?
5. Why are ReLU activations used?
6. Why are multiple linear layers without activations limited?
7. What are logits?
8. When do we use softmax?
9. Why should softmax not be applied before `CrossEntropyLoss`?
10. How many logits are needed for a 5-class problem?
11. What target shape does standard `CrossEntropyLoss` expect?
12. What target dtype is typically used?
13. What is a confusion matrix?
14. What is a decision boundary?
15. Why can MLP decision boundaries be nonlinear?
16. What is underfitting?
17. What is overfitting?
18. How can dropout help?
19. How can weight decay help?
20. Why should validation performance guide model improvements?


# Next Notebook

# 15 — Convolutional Neural Network Foundations

In the next notebook, we will study:

- Why MLPs are limited for images
- Image tensor structure
- What is convolution?
- Filters / kernels
- Sliding-window intuition
- Channels
- Feature maps
- `nn.Conv2d`
- Kernel size
- Stride
- Padding
- Output-size formula
- Pooling
- `nn.MaxPool2d`
- Flattening convolution features
- Building a small CNN
- Shape reasoning through a CNN
